# Step 2 - Database setup
MySQL connect karke `thdc_mentoring` database aur tables banana.

### 1. MySQL se connection check

In [2]:
from db import get_connection

conn = get_connection(use_database=False)
print("Connected to MySQL", conn.server_info)
conn.close()

Connected to MySQL 8.0.44


### 2. schema.sql chala kar database + tables banana

In [3]:
conn = get_connection(use_database=False)
cur = conn.cursor()

with open("schema.sql", encoding="utf-8") as f:
    sql = f.read()

for statement in sql.split(";"):
    if statement.strip():
        cur.execute(statement)

conn.commit()
conn.close()
print("database and tables created")

database and tables created


### 3. Tables check karna

In [4]:
from db import query

query("SHOW TABLES")

[{'Tables_in_thdc_mentoring': 'mentor_remarks'},
 {'Tables_in_thdc_mentoring': 'mentoring_forms'},
 {'Tables_in_thdc_mentoring': 'students'},
 {'Tables_in_thdc_mentoring': 'users'}]

In [5]:
import pandas as pd

pd.DataFrame(query("DESCRIBE students"))

,Field,Type,Null,Key,Default,Extra
0,id,int,NO,PRI,None,auto_increment
1,user_id,int,NO,UNI,None,
2,roll_no,varchar(30),YES,UNI,None,
3,branch,varchar(60),YES,,None,
4,class_name,varchar(40),YES,,None,
5,admitted_year,int,YES,,None,
6,mentor_id,int,YES,MUL,None,
7,cgpa,"decimal(4,2)",YES,,None,
8,backlogs,int,YES,,0,
9,placement_status,varchar(40),YES,,None,


### 4. Demo users daalna (testing ke liye)
Sab ka password `.env` ki `DEMO_PASSWORD` hai. Password hash karke save hota hai, plain text nahi.

In [6]:
import os
from werkzeug.security import generate_password_hash
from db import execute

demo_hash = generate_password_hash(os.getenv("DEMO_PASSWORD"))

people = [
    ("Director", "director@thdc.test", "director"),
    ("Dr. Amit Rawat", "faculty1@thdc.test", "faculty"),
    ("Dr. Neha Bisht", "faculty2@thdc.test", "faculty"),
    ("Rahul Negi", "student1@thdc.test", "student"),
    ("Priya Semwal", "student2@thdc.test", "student"),
    ("Ankit Panwar", "student3@thdc.test", "student"),
]

for name, email, role in people:
    if not query("SELECT id FROM users WHERE email = %s", (email,), one=True):
        execute("INSERT INTO users (name, email, password_hash, role) VALUES (%s, %s, %s, %s)",
                (name, email, demo_hash, role))

pd.DataFrame(query("SELECT id, name, email, role FROM users"))

,id,name,email,role
0,1,Director,director@thdc.test,director
1,2,Dr. Amit Rawat,faculty1@thdc.test,faculty
2,3,Dr. Neha Bisht,faculty2@thdc.test,faculty
3,4,Rahul Negi,student1@thdc.test,student
4,5,Priya Semwal,student2@thdc.test,student
5,6,Ankit Panwar,student3@thdc.test,student


### 5. Students ko roll no. aur mentor dena

In [7]:
def user_id(email):
    return query("SELECT id FROM users WHERE email = %s", (email,), one=True)["id"]

f1, f2 = user_id("faculty1@thdc.test"), user_id("faculty2@thdc.test")

students = [
    ("student1@thdc.test", "230101", "CSE", "B.Tech 3rd Year", 2023, f1),
    ("student2@thdc.test", "230102", "CSE", "B.Tech 3rd Year", 2023, f1),
    ("student3@thdc.test", "230201", "ECE", "B.Tech 3rd Year", 2023, f2),
]

for email, roll, branch, cls, year, mentor in students:
    uid = user_id(email)
    if not query("SELECT id FROM students WHERE user_id = %s", (uid,), one=True):
        execute("INSERT INTO students (user_id, roll_no, branch, class_name, admitted_year, mentor_id) "
                "VALUES (%s, %s, %s, %s, %s, %s)", (uid, roll, branch, cls, year, mentor))

### 6. JOIN - kaunsa student kis mentor ke paas hai

In [8]:
pd.DataFrame(query("""
    SELECT s.roll_no, u.name AS student, s.branch, m.name AS mentor
    FROM students s
    JOIN users u ON u.id = s.user_id
    LEFT JOIN users m ON m.id = s.mentor_id
    ORDER BY s.roll_no
"""))

,roll_no,student,branch,mentor
0,230101,Rahul Negi,CSE,Dr. Amit Rawat
1,230102,Priya Semwal,CSE,Dr. Amit Rawat
2,230201,Ankit Panwar,ECE,Dr. Neha Bisht
